# TDF Network analysis
In this notebook, we apply PageRank on our computed graphs while also visualizing our results.

## 1. Read graphs

In [ ]:
import networkx as nx
from pyvis.network import Network
import igraph as ig
import leidenalg
import networkx as nx
from collections import defaultdict

def to_number(x):
    try:
        return int(x)
    except ValueError:
        return float(x)

def read_pajek(name, path = "."):
    names = dict()
    G = nx.MultiDiGraph()
    with open(path + "/" + name + ".net", 'r', encoding="utf-8") as file:
        file.readline()

        for line in file:
            if line.startswith("*"):
                break
            else:
                node = line.split("\"")
                G.add_node(int(node[0]) - 1, label = node[1])
                names[int(node[0]) - 1] = node[1]

        for line in file:
            i, j, w = map(to_number, line.split())
            i -= 1
            j -= 1
            G.add_edge(i, j, weight=float(w))
      
    return G, names

def top10(data, top = 10):
    data = iter(data)
    for i in range(top):
        row = next(data)
        print(f"{i+1} - {row}")



graphs = dict()
name_mapping = dict()
digraphs = dict()
modes = ["no_weights", "time_diff", "normalized_time_diff", "scaled_time_diff", "points", "pure_points"]
for mode in modes:
    graphs[mode], name_mapping[mode] = read_pajek(f"TDF_{mode}", "output_graphs")
    print(f"Graph Info: {mode}")


Graph Info: no_weights
Graph Info: time_diff
Graph Info: normalized_time_diff
Graph Info: scaled_time_diff
Graph Info: points
Graph Info: pure_points


## 2. Compute PageRank for all graphs

In [2]:
results = dict()
for mode in modes:
    result = nx.pagerank(graphs[mode],max_iter=10000)
    results[mode] = {name: score for name,score in zip(name_mapping[mode].values(),result.values())}
    results[mode] = {k:v for k, v in sorted(results[mode].items(), key=lambda x: x[1], reverse=True)}


PageRank scores can be combined

In [14]:
combined_scores = {k: 0 for k in results["points"].keys()}

for name, score in results["points"].items():
    combined_scores[name] += score

for name, score in results["normalized_time_diff"].items():
    combined_scores[name] -= score

sorted_scores = dict(sorted(combined_scores.items(), key=lambda item: item[1], reverse=True))

In [15]:
top10(sorted_scores)

1 - ZABEL Erik
2 - CAVENDISH Mark
3 - SAGAN Peter
4 - FRANTZ Nicolas
5 - LEDUCQ André
6 - POGAČAR Tadej
7 - MERCKX Eddy
8 - MCEWEN Robbie
9 - BOTTECCHIA Ottavio
10 - PHILIPSEN Jasper


## 3. Visualize the results
For visualization we use the Pyvis library together with the Fruchterman-Reingold layout.

In [6]:
top_names = {mode: list(results[mode].keys())[:2000] for mode in modes}

best_result = {mode: list(results[mode].values())[0] for mode in modes}
worst_result = {mode: list(results[mode].values())[2000] for mode in modes}

keep_nodes = {mode: { node for node, data in graphs[mode].nodes(data=True) if data.get("label") in top_names[mode] } for mode in modes}

G_top = {mode: graphs[mode].subgraph(keep_nodes[mode]).copy() for mode in modes}

In [ ]:
def sizeMap(node,max,min,high,low):
    perc = (node - min) / (max - min)
    return (perc ** 2) * (high - low) + low

def multidigraph_to_digraph(G_multi):
    G_di = nx.DiGraph()
    
    for n, attr in G_multi.nodes(data=True):
        G_di.add_node(n, **attr)

    edge_attr_agg = defaultdict(lambda: {"weight": 0})
    max_weight = 0

    for u, v, data in G_multi.edges(data=True):
        edge_key = (u, v)
        weight = data.get("weight", 1)
        edge_attr_agg[edge_key]["weight"] += weight

        if edge_attr_agg[edge_key]["weight"] > max_weight:
            max_weight = edge_attr_agg[edge_key]["weight"]

        for k, v_attr in data.items():
            if k != "weight":
                edge_attr_agg[edge_key][k] = v_attr

    for (u, v), attr in edge_attr_agg.items():
        G_di.add_edge(u, v, **attr)

    return G_di, max_weight

digraph = dict()
maxWeight = dict()

for mode in modes:
    nodes = list(G_top[mode].nodes())
    idx_map = {n: i for i, n in enumerate(nodes)}
    edges_idx = [(idx_map[u], idx_map[v]) for u, v in G_top[mode].edges()]

    ig_g = ig.Graph(n=len(nodes), edges=edges_idx, directed=G_top[mode].is_directed())
    ig_g.vs['name'] = nodes

    partition_type = leidenalg.RBConfigurationVertexPartition
    
    partition = leidenalg.find_partition(
        ig_g,
        partition_type,
        resolution_parameter=1.0
    )
    membership = partition.membership

    for i, (node, data) in enumerate(G_top[mode].nodes(data=True)):
        G_top[mode].nodes[node]["group"] = membership[i]    
        G_top[mode].nodes[node]["size"] = sizeMap(results[mode][data["label"]], best_result[mode], worst_result[mode], 18, 3)
    digraph[mode], maxWeight[mode] = multidigraph_to_digraph(G_top[mode])

In [ ]:
for mode in modes:
    pos = nx.fruchterman_reingold_layout(digraph[mode])
    nt = Network('900px', '1500px', notebook=True)
    nt.toggle_physics(False)

    print("Layout calculated")

    for node in digraph[mode].nodes(data=True):
        n_id = node[0]
        n_data = node[1]
        x, y = pos[n_id]
        nt.add_node(
            n_id,
            label = str(n_data["label"]) if n_data.get("label") in list(results["points"])[0:30] else "",
            x = x * 820,  # scale to make layout visible
            y = y * 820,
            title = n_data["label"],
            group = n_data["group"],
            size = n_data["size"],
            fixed = True  # lock the position
        )
    
    print("Node positions calculated")

    edge_list = []
    for u, v, data in digraph[mode].edges(data=True):
        weight = data.get("weight", 1)
        alpha = min(1.0, weight / maxWeight[mode])

        edge_list.append({
            "from": u,
            "to": v,
            "width": 0.5,
            "color": f"rgba(70, 70, 70, {alpha})"
        })

    nt.edges = edge_list

    print("Edges calculated")

    nt.show(f'nx-{mode}.html')

Layout calculated
Node positions calculated
Edges calculated
nx-no_weights.html
Layout calculated
Node positions calculated
Edges calculated
nx-time_diff.html
Layout calculated
Node positions calculated
Edges calculated
nx-normalized_time_diff.html
Layout calculated
Node positions calculated
Edges calculated
nx-scaled_time_diff.html
Layout calculated
Node positions calculated
Edges calculated
nx-points.html
Layout calculated
Node positions calculated
Edges calculated
nx-pure_points.html


## 4. In-depth analysis of PageRank results

In [ ]:
top10(results["no_weights"], 20)
"""
Leducq, 2 times GC winner 9 time contender, 25 stage winner, 2x 2nd GC, 3x 3rd GC
Garrigou, 1 time GC winner, 8 time contender, 8 time stage winner, 6 time contender, 3x 2nd GC, 2x 3rd GC
Alavoine, 11 time contender, 2x 2nd GC, 17 stages, 2x 3rd GC
Frantz, 2 times GC winner, 7 time contender, 20 stages, 2x 2nd GC
Magne, 2 times GC winner, 10 time contender, 10 stages, 1x 2nd GC
Thys, 3 time GC winner, 10 time contender, 13 stages
"""

1 - LEDUCQ André
2 - GARRIGOU Gustave
3 - ALAVOINE Jean
4 - FRANTZ Nicolas
5 - MAGNE Antonin
6 - THYS Philippe
7 - FABER François
8 - CHRISTOPHE Eugène
9 - LAMBOT Firmin
10 - GEORGET Émile
11 - TIBERGHIEN Hector
12 - PETIT-BRETON Lucien
13 - BELLENGER Romain
14 - MOTTIAT Louis
15 - BUYSSE Lucien
16 - ZOETEMELK Joop
17 - REBRY Gaston
18 - TROUSSELIER Louis
19 - DARRIGADE André
20 - POULIDOR Raymond


In [ ]:
top10(results["points"], 20)

"""
Highlights many GC contenders and individuals with many stage victories such as Eddy Merckx, Erik Zabel,
Peter Sagan, Mark Cavendish.
"""

1 - FRANTZ Nicolas
2 - LEDUCQ André
3 - THYS Philippe
4 - ZABEL Erik
5 - MERCKX Eddy
6 - FABER François
7 - SAGAN Peter
8 - GARRIGOU Gustave
9 - ALAVOINE Jean
10 - CAVENDISH Mark
11 - POGAČAR Tadej
12 - BOTTECCHIA Ottavio
13 - KELLY Sean
14 - PÉLISSIER Henri
15 - PÉLISSIER Charles
16 - MAGNE Antonin
17 - DARRIGADE André
18 - HINAULT Bernard
19 - PETIT-BRETON Lucien
20 - BELLENGER Romain


In [ ]:
top10(results["pure_points"], 20)

"""
Better highlights sprinters.
"""

1 - FRANTZ Nicolas
2 - LEDUCQ André
3 - ZABEL Erik
4 - SAGAN Peter
5 - MERCKX Eddy
6 - THYS Philippe
7 - POGAČAR Tadej
8 - CAVENDISH Mark
9 - FABER François
10 - GARRIGOU Gustave
11 - ALAVOINE Jean
12 - KELLY Sean
13 - BOTTECCHIA Ottavio
14 - PÉLISSIER Charles
15 - HINAULT Bernard
16 - PÉLISSIER Henri
17 - DARRIGADE André
18 - MCEWEN Robbie
19 - MAGNE Antonin
20 - VAN AERT Wout


In [ ]:
top10(results["time_diff"], 20)

"""
Highlights elite climbers and GC contenders that can leave a large gap between other riders and theirselves.
"""

1 - GARRIGOU Gustave
2 - FABER François
3 - CHRISTOPHE Eugène
4 - FRANTZ Nicolas
5 - ALAVOINE Jean
6 - THYS Philippe
7 - LAMBOT Firmin
8 - MAGNE Antonin
9 - LEDUCQ André
10 - PETIT-BRETON Lucien
11 - ZOETEMELK Joop
12 - VAN IMPE Lucien
13 - GEORGET Émile
14 - POULIDOR Raymond
15 - TROUSSELIER Louis
16 - DELGADO Pedro
17 - BUYSSE Lucien
18 - TIBERGHIEN Hector
19 - DEWAELE Maurice
20 - AGOSTINHO Joaquim


In [10]:
top10(results["normalized_time_diff"], 20)

1 - GARRIGOU Gustave
2 - CHRISTOPHE Eugène
3 - MAGNE Antonin
4 - THYS Philippe
5 - LEDUCQ André
6 - ALAVOINE Jean
7 - FABER François
8 - FRANTZ Nicolas
9 - ZOETEMELK Joop
10 - LAMBOT Firmin
11 - TIBERGHIEN Hector
12 - POULIDOR Raymond
13 - DEFRAEYE Odiel
14 - VAN IMPE Lucien
15 - HEUSGHEM Louis
16 - REBRY Gaston
17 - AGOSTINHO Joaquim
18 - MERCKX Eddy
19 - BUYSSE Marcel
20 - PETIT-BRETON Lucien


In [12]:
top10(results["pure_points"], 20)

1 - FRANTZ Nicolas
2 - LEDUCQ André
3 - ZABEL Erik
4 - SAGAN Peter
5 - MERCKX Eddy
6 - THYS Philippe
7 - POGAČAR Tadej
8 - CAVENDISH Mark
9 - FABER François
10 - GARRIGOU Gustave
11 - ALAVOINE Jean
12 - KELLY Sean
13 - BOTTECCHIA Ottavio
14 - PÉLISSIER Charles
15 - HINAULT Bernard
16 - PÉLISSIER Henri
17 - DARRIGADE André
18 - MCEWEN Robbie
19 - MAGNE Antonin
20 - VAN AERT Wout


In [13]:
top10(results["scaled_time_diff"], 20)

1 - GARRIGOU Gustave
2 - CHRISTOPHE Eugène
3 - MAGNE Antonin
4 - THYS Philippe
5 - LEDUCQ André
6 - ALAVOINE Jean
7 - FABER François
8 - FRANTZ Nicolas
9 - LAMBOT Firmin
10 - TIBERGHIEN Hector
11 - ZOETEMELK Joop
12 - DEFRAEYE Odiel
13 - POULIDOR Raymond
14 - VAN IMPE Lucien
15 - REBRY Gaston
16 - HEUSGHEM Louis
17 - BUYSSE Marcel
18 - PETIT-BRETON Lucien
19 - DEWAELE Maurice
20 - GEORGET Émile
